# Aiyagari Model with Unemployment Insurance

**Course:** Macroeconomics — Incomplete Markets  
**Authors:** Sneha Thankkam Raju, Suryashis Ghosh  
**Date:** May 2025

---

## Model Overview

This notebook implements an Aiyagari (1994) style incomplete-markets model extended with:
- A two-state Markov employment process (employed / unemployed)
- Unemployment insurance financed by a proportional wage tax
- A balanced government budget in every period

We solve for the stationary general equilibrium at three tax rates: **t = 10%, 20%, 30%**,
and compare aggregate capital levels and the stationary wealth distribution.

### Key Theoretical Results

**Unemployment benefit** (from balanced budget + stationary Markov distribution):
$$b = \frac{8}{3} \cdot t \cdot w$$

**Agent's Bellman equation:**
$$v(k, s) = \max_{k'} \left\{ \ln\left[ rk + ws + \frac{8t}{3}w(1-s) - k' + (1-\delta)k + \theta \right] + \beta \sum_{s'} P_{ss'} v(k', s') \right\}$$

## 0. Setup

Install `quantecon` if not already present (provides efficient discrete dynamic programming solvers).

In [ ]:
# Install quantecon for DiscreteDP policy iteration solver
# Only needs to run once; comment out after first install
%pip install quantecon -q

## 1. Imports & Global Parameters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numba import jit
from scipy.optimize import root_scalar
from quantecon.markov import DiscreteDP

# ── Production & Preferences ──────────────────────────────────────────────────
A = 1.0      # Total factor productivity (Cobb-Douglas)
α = 0.33     # Capital share in production  (Y = A * K^α * L^(1-α))
β = 0.96     # Household discount factor
δ = 0.075    # Capital depreciation rate per period
θ = 1e-5     # Small smoothing constant in log utility: ln(c + θ)
             # Prevents log(0) when consumption hits the constraint

# ── Idiosyncratic Employment States ──────────────────────────────────────────
# z = 0: unemployed (earns unemployment benefit only)
# z = 1: employed   (earns after-tax wage w*(1-t))
z_vals = np.array([0.0, 1.0])

# ── Markov Transition Matrix ──────────────────────────────────────────────────
# Π[i, j] = P(next state = j | current state = i)
#
#              Unemp(t+1)  Emp(t+1)
# Unemp(t)  [   0.2         0.8   ]
# Emp(t)    [   0.3         0.7   ]
#
# → Stationary distribution: π_U = 3/11,  π_E = 8/11
Π = np.array([[0.2, 0.8],
              [0.3, 0.7]])

## 2. Reward & Transition Array Builders

The state space is indexed as `s_i = a_i * z_size + z_i`, so employment status varies
within each asset level (column-minor order). We use `@jit` (Numba) to speed up the
nested loops that populate the `(n × a_size)` reward matrix `R` and
the `(n × a_size × n)` transition tensor `Q`.

In [ ]:
@jit(nopython=True)
def populate_R(R, a_size, z_size, a_vals, z_vals, r, w, t):
    """
    Fill the reward matrix R[s_i, a_new_i] with per-period utility.

    State index encoding:  s_i = a_i * z_size + z_i
      - a_i   : index into asset grid a_vals
      - z_i   : employment state index (0=unemp, 1=emp)

    Budget constraint:
      c = w*(1-t)*z + b*(1-z) + (1+r)*a - a'
      where b = (8/3)*t*w  (derived from balanced budget & stationary dist.)

    Utility: ln(c + θ)   if c > 0
             -inf         otherwise  (infeasible choice)
    """
    n = a_size * z_size
    for s_i in range(n):
        a_i = s_i // z_size          # current asset index
        z_i = s_i % z_size           # current employment state index
        a   = a_vals[a_i]            # current asset level
        z   = z_vals[z_i]            # employment status (0 or 1)

        for new_a_i in range(a_size):
            a_new = a_vals[new_a_i]  # next-period asset choice

            w_income = w * z * (1 - t)        # after-tax labour income (0 if unemployed)
            b        = (8.0 / 3.0) * t * w    # unemployment benefit (balanced budget)
            c        = w_income + b * (1 - z) + (1 + r) * a - a_new

            if c > 0:
                R[s_i, new_a_i] = np.log(c + θ)
            else:
                R[s_i, new_a_i] = -np.inf     # negative consumption → infeasible


@jit(nopython=True)
def populate_Q(Q, a_size, z_size, Π):
    """
    Fill the transition probability tensor Q[s_i, a_new_i, s_new_i].

    Given current state (a_i, z_i) and choice a_new_i, the next state
    is (a_new_i, z_new_i) with probability Π[z_i, z_new_i].
    Asset choice is deterministic; only employment status is stochastic.
    """
    n = a_size * z_size
    for s_i in range(n):
        z_i = s_i % z_size                        # current employment state
        for a_new_i in range(a_size):
            for next_z_i in range(z_size):        # iterate over next-period z
                s_new_i = a_new_i * z_size + next_z_i
                Q[s_i, a_new_i, s_new_i] = Π[z_i, next_z_i]


@jit(nopython=True)
def asset_marginal(s_probs, a_size, z_size):
    """
    Collapse the joint stationary distribution over (a, z) to the
    marginal distribution over assets only, by summing out employment states.

    Parameters
    ----------
    s_probs : array of shape (a_size * z_size,)
        Stationary distribution over joint states.

    Returns
    -------
    a_probs : array of shape (a_size,)
        Marginal probability mass on each asset grid point.
    """
    a_probs = np.zeros(a_size)
    for a_i in range(a_size):
        for z_i in range(z_size):
            a_probs[a_i] += s_probs[a_i * z_size + z_i]
    return a_probs

## 3. Household Class

Encapsulates the agent's problem for given prices `(r, w)` and policy parameter `t`.

In [ ]:
class Household:
    """
    Represents the household optimisation problem.

    Asset grid: uniformly spaced from a_min to a_max with a_size points.
    Lower bound a_min > 0 enforces a strict no-borrowing constraint.

    Attributes
    ----------
    r, w, t  : prices and tax rate (set externally by equilibrium loop)
    a_vals   : asset grid (shape: a_size)
    R        : reward matrix  (shape: n × a_size)
    Q        : transition tensor (shape: n × a_size × n)
    """

    def __init__(self, r, w, t, a_min=1e-10, a_max=25, a_size=200):
        self.r, self.w, self.t = r, w, t

        # Asset grid — a_min > 0 acts as the borrowing constraint
        self.a_vals  = np.linspace(a_min, a_max, a_size)
        self.a_size  = a_size
        self.z_vals  = z_vals
        self.z_size  = len(z_vals)
        self.n       = a_size * self.z_size   # total number of joint states

        # Initialise arrays (populated by build)
        self.R = np.empty((self.n, a_size))
        self.Q = np.zeros((self.n, a_size, self.n))
        self.build()

    def build(self):
        """Populate R and Q using Numba-compiled helpers."""
        populate_R(self.R, self.a_size, self.z_size,
                   self.a_vals, self.z_vals, self.r, self.w, self.t)
        populate_Q(self.Q, self.a_size, self.z_size, Π)

## 4. Firm-Side Pricing & Equilibrium Solver

From profit maximisation by a representative firm with CRS Cobb-Douglas technology
`Y = A * K^α * L^(1-α)`, the competitive factor prices are:

$$r = A\alpha K^{\alpha-1} - \delta \qquad w = A(1-\alpha)\left(\frac{A\alpha}{r+\delta}\right)^{\alpha/(1-\alpha)}$$

We use `r_to_w` to get the implied wage given `r`, and `prices_to_K` to solve the
household problem and compute aggregate capital demand from the stationary distribution.

In [ ]:
def r_to_w(r):
    """
    Compute the competitive equilibrium wage given interest rate r.

    Derived from firm's first-order conditions:
        w = A*(1-α) * (A*α / (r+δ))^(α/(1-α))
    """
    return A * (1 - α) * (A * α / (r + δ)) ** (α / (1 - α))


def prices_to_K(r, t):
    """
    Given interest rate r and tax rate t:
      1. Compute equilibrium wage w
      2. Solve household DP via policy iteration
      3. Find stationary distribution over (a, z)
      4. Return aggregate capital K = E[a], marginal asset dist., and asset grid
    """
    w  = r_to_w(r)
    hh = Household(r, w, t)

    # Solve with policy iteration (faster than value iteration for this problem)
    ddp     = DiscreteDP(hh.R, hh.Q, β)
    results = ddp.solve(method='policy_iteration')

    # Extract the unique stationary distribution of the implied Markov chain
    stationary_probs = results.mc.stationary_distributions[0]

    # Marginalise over employment states to get the asset distribution
    a_probs = asset_marginal(stationary_probs, hh.a_size, hh.z_size)

    # Aggregate capital = expected assets under the stationary distribution
    K = np.sum(a_probs * hh.a_vals)
    return K, a_probs, hh.a_vals


def capital_gap(r, t, K_target):
    """
    Excess capital supply: used as the objective for the equilibrium root-finder.
    We search for r* such that K(r*) = K_target  (market clearing).
    """
    K, *_ = prices_to_K(r, t)
    return K - K_target

## 5. Solve for General Equilibrium at Each Tax Rate

We target aggregate capital levels corresponding to the three tax rates and use
Brent's method (`root_scalar`) to find the market-clearing interest rate `r*`.

In [ ]:
# Target K values chosen to approximately correspond to each tax rate's equilibrium
# (from prior calibration; the solver confirms/refines them)
desired_Ks  = [6.0, 4.5, 3.0]
tax_rates   = [0.10, 0.20, 0.30]

# Storage for results
found_rs        = []
found_Ks        = []
a_distributions = []

for t, K_target in zip(tax_rates, desired_Ks):
    print(f"Solving for tax rate t = {int(t*100)}% ...")

    # Brent's method: bracket must contain a sign change in capital_gap
    # r ∈ (0.001, 0.08) is a safe range for plausible parameterisations
    sol    = root_scalar(capital_gap, args=(t, K_target),
                         bracket=[0.001, 0.08], method='brentq')
    r_star = sol.root

    # Recover equilibrium objects at the market-clearing rate
    K, dist, a_vals = prices_to_K(r_star, t)

    found_rs.append(r_star)
    found_Ks.append(K)
    a_distributions.append(dist)

    print(f"  → r* = {r_star:.4f},  K* = {K:.4f}")

print("\nDone.")

## 6. Plot: Stationary Capital Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for dist, t in zip(a_distributions, tax_rates):
    ax.plot(a_vals, dist, lw=2, label=f't = {int(t*100)}%')

ax.set_xlabel("Capital (k)", fontsize=13)
ax.set_ylabel("Density", fontsize=13)
ax.set_title("Distribution of Capital Across Agents for Different Tax Rates", fontsize=14)
ax.legend(fontsize=12)
ax.grid(alpha=0.3)
plt.tight_layout()

# Save figure for repository
plt.savefig("../figures/capital_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

## 7. Summary Table

In [ ]:
print(f"{'Tax Rate':>10} {'Agg. Capital K':>16} {'Equilibrium r':>15}")
print("-" * 45)
for t, K, r in zip(tax_rates, found_Ks, found_rs):
    print(f"  {int(t*100):>3}%       {K:>10.4f}         {r:>10.4f}")

### Interpretation

- **Higher taxes → lower aggregate capital**: A higher tax rate reduces after-tax wages, weakening the precautionary savings motive. More generous unemployment benefits simultaneously reduce the urgency to self-insure.
- **Higher taxes → higher equilibrium r**: With less capital in the economy, the marginal product of capital (and hence the interest rate) rises.
- The distribution shifts toward **zero** as t increases, with a sharper mass point at the borrowing constraint.